# DP-SGD: 差分隐私训练 + 成员推理攻击防御评估

## 项目简介

本项目从零实现 **DP-SGD (Differentially Private Stochastic Gradient Descent)**，并通过**成员推理攻击 (Membership Inference Attack, MIA)** 评估其隐私保护效果。

**核心故事线**：
1. 训练一个标准模型（无隐私保护）→ 发起成员推理攻击 → 观察隐私泄露
2. 训练 DP-SGD 模型（不同隐私强度）→ 发起同样攻击 → 观察防御效果
3. 可视化"隐私预算 - 模型精度 - 攻击成功率"三角权衡

**参考论文**：
- Abadi et al., "Deep Learning with Differential Privacy", CCS 2016
- Yeom et al., "Privacy Risk in Machine Learning", CSF 2018
- Mironov, "Renyi Differential Privacy", CSF 2017

## 1. 什么是差分隐私？

**差分隐私 (Differential Privacy)** 提供了一个数学框架，保证模型的输出不会泄露任何单个训练样本的信息。

**形式化定义**：对于任何两个仅相差一条记录的数据集 D 和 D'，满足：

$$P[M(D) \in S] \leq e^{\varepsilon} \cdot P[M(D') \in S] + \delta$$

其中：
- **ε (epsilon)** 是隐私预算，越小越私密
- **δ (delta)** 是松弛参数，通常取 1/n (n 为数据集大小)

## 2. DP-SGD 核心机制

DP-SGD 在标准 SGD 基础上增加三个步骤：

```
标准梯度 → ① 逐样本计算梯度 → ② 梯度裁剪 (L2范数≤C) → ③ 添加高斯噪声 N(0, σ²C²I) → 更新参数
```

- **逐样本梯度**：为每个训练样本独立计算梯度，而非整个 batch 的平均
- **梯度裁剪**：将每个样本梯度的 L2 范数裁剪到阈值 C，限制单个样本的影响
- **高斯噪声**：向聚合梯度添加噪声，模糊单个样本的贡献

## 3. 成员推理攻击 (MIA)

MIA 判断某个样本是否被用于训练模型。核心直觉：**过拟合的模型对训练数据的损失更低**。

- 对训练集样本（成员）和非训练样本（非成员）分别计算模型损失
- 成员的损失通常更低 → 可以用损失值作为区分信号
- 用 **AUC** 衡量攻击成功率：0.5 = 随机猜测（攻击失败），> 0.5 = 隐私泄露

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print("环境准备完成")

## 4. 数据加载与划分

使用 sklearn 内置的 digits 数据集（8x8 手写数字，1797 样本，无需下载）。

为了让模型更容易过拟合（从而使 MIA 效果更明显），我们使用较小的训练集（30%）和较大的测试集（70%）。

In [ ]:
digits = load_digits()
X = torch.tensor(digits.data, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)
X = X / 16.0  # 归一化到 [0, 1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.7, random_state=42, stratify=y
)

n_mia = min(250, len(X_train), len(X_test))
X_mia_members = X_train[:n_mia]
y_mia_members = y_train[:n_mia]
X_mia_nonmembers = X_test[:n_mia]
y_mia_nonmembers = y_test[:n_mia]

print(f"数据集: sklearn digits (8x8 手写数字, {len(digits.data)} 样本)")
print(f"训练集: {len(X_train)} 样本")
print(f"测试集(非成员): {len(X_test)} 样本")
print(f"MIA 评估: 各 {n_mia} 个成员/非成员样本")

## 5. 模型定义

使用一个参数量较大的 MLP（相对于训练集大小），有助于产生过拟合，从而让 MIA 效果更明显。

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

model = SimpleMLP()
n_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {n_params:,}")
print(f"参数/样本比: {n_params/len(X_train):.1f} (>1 表示模型有能力记忆训练数据)")
del model

## 6. DP-SGD 核心函数

以下是 DP-SGD 的三个核心操作：逐样本梯度计算、梯度裁剪、噪声注入。

In [ ]:
def compute_per_sample_grads(model, loss_fn, data, targets):
    """逐样本计算梯度 (教学写法: 显式循环)"""
    per_sample_grads = []
    for x_i, y_i in zip(data, targets):
        model.zero_grad()
        output = model(x_i.unsqueeze(0))
        loss = loss_fn(output, y_i.unsqueeze(0))
        loss.backward()
        sample_grad = []
        for param in model.parameters():
            sample_grad.append(param.grad.detach().clone().flatten())
        per_sample_grads.append(torch.cat(sample_grad))
    return torch.stack(per_sample_grads)


def clip_gradients(per_sample_grads, max_norm):
    """L2 范数裁剪: g_i * min(1, C / ||g_i||)"""
    norms = torch.norm(per_sample_grads, dim=1)
    clip_factors = torch.clamp(max_norm / (norms + 1e-8), max=1.0)
    return per_sample_grads * clip_factors.unsqueeze(1)


def aggregate_and_noise(clipped_grads, noise_multiplier, max_norm, batch_size):
    """聚合裁剪梯度并添加高斯噪声"""
    aggregated = clipped_grads.sum(dim=0)
    noise = torch.randn_like(aggregated) * (noise_multiplier * max_norm)
    return (aggregated + noise) / batch_size


def apply_gradient(model, flat_grad, lr):
    """将扁平梯度向量写回模型参数"""
    offset = 0
    with torch.no_grad():
        for param in model.parameters():
            numel = param.numel()
            param.data -= lr * flat_grad[offset:offset + numel].reshape(param.shape)
            offset += numel

print("DP-SGD 核心函数定义完成")

## 7. 隐私预算计算 (RDP)

基于 Renyi 差分隐私 (Mironov 2017) 追踪训练过程中的隐私预算消耗。

In [ ]:
def compute_rdp(q, noise_multiplier, steps, orders):
    """计算 Renyi 差分隐私"""
    rdp = []
    for alpha in orders:
        rdp_single = alpha / (2.0 * noise_multiplier ** 2)
        log_term = math.log1p(q * q * (math.exp(min(rdp_single, 500)) - 1))
        rdp_composed = steps * min(rdp_single, log_term / max(alpha - 1, 1e-10))
        rdp.append(rdp_composed)
    return rdp

def rdp_to_epsilon(rdp_values, orders, delta):
    """RDP -> (epsilon, delta)-DP 转换"""
    eps_list = []
    for rdp_val, alpha in zip(rdp_values, orders):
        eps = rdp_val - math.log(delta) / (alpha - 1)
        eps_list.append(eps)
    return min(eps_list)

def compute_epsilon(batch_size, dataset_size, noise_multiplier, epochs, delta=1e-5):
    """给定训练参数, 计算最终 epsilon"""
    q = batch_size / dataset_size
    steps = epochs * math.ceil(dataset_size / batch_size)
    orders = [1.5, 2, 2.5, 3, 4, 5, 6, 8, 16, 32, 64]
    rdp = compute_rdp(q, noise_multiplier, steps, orders)
    return rdp_to_epsilon(rdp, orders, delta)

print("隐私预算计算函数定义完成")

## 8. 训练函数 + 成员推理攻击

In [ ]:
def run_evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        preds = outputs.argmax(dim=1)
        return (preds == y).float().mean().item()

def train_standard(model, X_train, y_train, X_test, y_test, epochs=500, lr=0.001, batch_size=32):
    """标准训练 (无隐私保护)"""
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    dataset_size = len(X_train)
    for epoch in range(epochs):
        model.train()
        indices = torch.randperm(dataset_size)
        for start in range(0, dataset_size, batch_size):
            end = min(start + batch_size, dataset_size)
            batch_idx = indices[start:end]
            optimizer.zero_grad()
            output = model(X_train[batch_idx])
            loss = loss_fn(output, y_train[batch_idx])
            loss.backward()
            optimizer.step()
        if (epoch + 1) % (epochs // 5) == 0:
            train_acc = run_evaluate(model, X_train, y_train)
            test_acc = run_evaluate(model, X_test, y_test)
            print(f"  Epoch {epoch+1}/{epochs}  Train: {train_acc:.4f}  Test: {test_acc:.4f}")
    return run_evaluate(model, X_test, y_test)

def train_dpsgd(model, X_train, y_train, X_test, y_test,
                epochs=30, lr=0.05, max_norm=1.0, noise_multiplier=1.0, batch_size=64):
    """DP-SGD 训练"""
    loss_fn = nn.CrossEntropyLoss()
    dataset_size = len(X_train)
    delta = 1.0 / dataset_size
    for epoch in range(epochs):
        model.train()
        indices = torch.randperm(dataset_size)
        for start in range(0, dataset_size, batch_size):
            end = min(start + batch_size, dataset_size)
            batch_idx = indices[start:end]
            X_batch, y_batch = X_train[batch_idx], y_train[batch_idx]
            per_grads = compute_per_sample_grads(model, loss_fn, X_batch, y_batch)
            clipped = clip_gradients(per_grads, max_norm)
            noised = aggregate_and_noise(clipped, noise_multiplier, max_norm, len(X_batch))
            apply_gradient(model, noised, lr)
        if (epoch + 1) % 10 == 0:
            acc = run_evaluate(model, X_test, y_test)
            eps = compute_epsilon(batch_size, dataset_size, noise_multiplier, epoch + 1, delta)
            print(f"  Epoch {epoch+1}/{epochs}  Acc: {acc:.4f}  epsilon={eps:.2f}")
    final_eps = compute_epsilon(batch_size, dataset_size, noise_multiplier, epochs, delta)
    final_acc = run_evaluate(model, X_test, y_test)
    return final_acc, final_eps

def membership_inference_attack(model, X_members, y_members, X_nonmembers, y_nonmembers):
    """成员推理攻击: 基于损失值区分成员/非成员"""
    model.eval()
    with torch.no_grad():
        loss_m = F.cross_entropy(model(X_members), y_members, reduction='none').numpy()
        loss_nm = F.cross_entropy(model(X_nonmembers), y_nonmembers, reduction='none').numpy()
    labels = np.concatenate([np.ones(len(loss_m)), np.zeros(len(loss_nm))])
    scores = np.concatenate([-loss_m, -loss_nm])  # 负损失: 越低越可能是成员
    auc = roc_auc_score(labels, scores)
    return auc, loss_m, loss_nm

print("训练函数和攻击函数定义完成")

## 9. 实验 1: 标准模型的隐私漏洞

训练一个标准模型（无隐私保护），然后对其发起成员推理攻击。
模型训练 500 轮后会严重过拟合，对训练数据产生"记忆"效应。

In [ ]:
torch.manual_seed(42)
model_std = SimpleMLP()
print("训练标准模型 (无隐私保护)...")
acc_std = train_standard(model_std, X_train, y_train, X_test, y_test, epochs=500, lr=0.001, batch_size=32)

auc_std, loss_m_std, loss_nm_std = membership_inference_attack(
    model_std, X_mia_members, y_mia_members, X_mia_nonmembers, y_mia_nonmembers
)
print(f"\n结果: 准确率={acc_std:.4f}, MIA AUC={auc_std:.4f}")
print(f"MIA AUC > 0.5 说明攻击者能区分成员和非成员 → 隐私泄露!")

## 10. 实验 2: DP-SGD 防御效果

使用不同噪声水平的 DP-SGD 训练模型，观察：
- 噪声越大 → 隐私保护越强 (epsilon 越小)
- 噪声越大 → MIA AUC 越接近 0.5 (攻击失败)
- 噪声越大 → 模型准确率下降 (隐私-效用权衡)

In [ ]:
results = [("标准训练", acc_std, auc_std, float('inf'), 0)]
dp_losses = {}

for sigma in [1.0, 3.0, 5.0]:
    print(f"\n--- DP-SGD (sigma={sigma}) ---")
    torch.manual_seed(42)
    model_dp = SimpleMLP()
    acc_dp, eps_dp = train_dpsgd(
        model_dp, X_train, y_train, X_test, y_test,
        epochs=30, lr=0.05, max_norm=1.0, noise_multiplier=sigma, batch_size=64
    )
    auc_dp, loss_m_dp, loss_nm_dp = membership_inference_attack(
        model_dp, X_mia_members, y_mia_members, X_mia_nonmembers, y_mia_nonmembers
    )
    print(f"  准确率: {acc_dp:.4f} | epsilon: {eps_dp:.2f} | MIA AUC: {auc_dp:.4f}")
    results.append((f"DP-SGD(sigma={sigma})", acc_dp, auc_dp, eps_dp, sigma))
    dp_losses[sigma] = (loss_m_dp, loss_nm_dp)

print("\n" + "=" * 55)
print(f"{'训练方式':<20} {'准确率':>8} {'MIA AUC':>10} {'epsilon':>10}")
print("-" * 55)
for name, acc, auc, eps, _ in results:
    eps_str = "inf" if eps == float('inf') else f"{eps:.2f}"
    print(f"{name:<20} {acc:>8.4f} {auc:>10.4f} {eps_str:>10}")

## 11. 可视化分析

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 统一 x 轴范围
all_losses = np.concatenate([loss_m_std, loss_nm_std])
best_sigma = max(dp_losses.keys())
loss_m_dp, loss_nm_dp = dp_losses[best_sigma]
all_losses_dp = np.concatenate([loss_m_dp, loss_nm_dp])
x_max = max(np.percentile(all_losses, 99), np.percentile(all_losses_dp, 99))
bins = np.linspace(0, x_max, 30)

# 图 1: 标准模型 — 成员损失集中在 0 附近, 非成员更分散
ax1 = axes[0]
ax1.hist(loss_m_std, bins=bins, alpha=0.6, label='Members', color='#e74c3c', density=True)
ax1.hist(loss_nm_std, bins=bins, alpha=0.6, label='Non-members', color='#3498db', density=True)
ax1.set_title('Standard Training\n(No Privacy)')
ax1.set_xlabel('Per-sample Loss')
ax1.set_ylabel('Density')
ax1.set_xlim(0, x_max)
ax1.legend()

# 图 2: DP-SGD 模型 — 成员和非成员分布重叠
ax2 = axes[1]
ax2.hist(loss_m_dp, bins=bins, alpha=0.6, label='Members', color='#e74c3c', density=True)
ax2.hist(loss_nm_dp, bins=bins, alpha=0.6, label='Non-members', color='#3498db', density=True)
ax2.set_title(f'DP-SGD (sigma={best_sigma})\n(Strong Privacy)')
ax2.set_xlabel('Per-sample Loss')
ax2.set_ylabel('Density')
ax2.set_xlim(0, x_max)
ax2.legend()

# 图 3: 权衡曲线
ax3 = axes[2]
dp_r = [(n, a, u, e) for n, a, u, e, s in results if s > 0]
if dp_r:
    epsilons = [r[3] for r in dp_r]
    accs = [r[1] for r in dp_r]
    aucs = [r[2] for r in dp_r]
    ax3_twin = ax3.twinx()
    l1, = ax3.plot(epsilons, accs, 'o-', color='#2ecc71', linewidth=2, markersize=8, label='Accuracy')
    l2, = ax3_twin.plot(epsilons, aucs, 's-', color='#e74c3c', linewidth=2, markersize=8, label='MIA AUC')
    ax3_twin.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax3.set_xlabel('Privacy Budget (epsilon)')
    ax3.set_ylabel('Model Accuracy', color='#2ecc71')
    ax3_twin.set_ylabel('MIA AUC', color='#e74c3c')
    ax3.set_title('Privacy-Utility-Attack\nTradeoff')
    ax3.legend([l1, l2], ['Accuracy', 'MIA AUC'], loc='center right')

plt.tight_layout()
plt.savefig('dp_sgd_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存: dp_sgd_results.png")

## 12. 结论与讨论

### 实验发现

1. **标准模型存在隐私泄露**：MIA AUC > 0.5，攻击者能在一定程度上区分训练数据成员
2. **DP-SGD 有效降低隐私风险**：随着噪声增大（sigma 增大），MIA AUC 逐渐接近 0.5
3. **隐私-效用权衡**：更强的隐私保护（更小的 epsilon）必然伴随模型准确率下降

### 为什么 digits 数据集上的 MIA 效果不够剧烈？

digits 数据集样本类别区分度高、模式清晰，模型可以通过学习通用特征（而非记忆个别样本）达到高准确率。在更复杂、更模糊的真实数据集上，MIA 攻击效果会更加显著。

### 实际应用建议

- **epsilon < 1**：强隐私保护，适合敏感数据（医疗、金融）
- **epsilon 1-10**：中等隐私保护，适合一般场景
- **epsilon > 10**：弱隐私保护，与无保护差别不大

### 参考文献

1. Abadi et al., "Deep Learning with Differential Privacy", CCS 2016
2. Yeom et al., "Privacy Risk in Machine Learning: Analyzing the Connection to Overfitting", CSF 2018
3. Mironov, "Renyi Differential Privacy", CSF 2017
4. Shokri et al., "Membership Inference Attacks Against Machine Learning Models", S&P 2017